In [18]:
import sys

sys.path.insert(0, "..")
from src.api import fetch_list, save_raw, fetch_details

# 1. 법령(eflaw)

In [86]:
EFLAW_QUERIES: list[str] = [
    "공인중개사",           # 중개계약, 중개수수료
    "집합건물",               # 집합건물의소유및관리에관한법률
    "주택",                 # 주택 공급/관리 기준
    "부동산",
    "임대",
    "임차",
    "전세",
    "보증금",
    "매도",
    "매수"
    
]

In [87]:
seen_eflaw: set[str] = set()  # 수집된 법령ID 추적
eflaw_items: list[dict] = []

for query in EFLAW_QUERIES:
    items = fetch_list("eflaw", query=query, max_items=None, extra_params={"nw": 3, "search": 2})
    for item in items:
        doc_id = str(item.get("법령ID", ""))
        # doc_id가 비어있거나 이미 수집한 항목이면 skip
        if doc_id and doc_id not in seen_eflaw:
            seen_eflaw.add(doc_id)
            item.pop("법령상세링크", None)  # 법령상세링크 컬럼 제외
            eflaw_items.append(item)

save_raw(eflaw_items, target="eflaw", mode="w")

print(len(eflaw_items))

1707


In [88]:
eflaw_details = fetch_details(
    target="eflaw",
    items=eflaw_items,
    id_field="법령ID",
)

# 목록만 있는 eflaw.jsonl 덮어쓰기
result = save_raw(eflaw_details, target="eflaw", mode="w")
print(len(eflaw_details))

1707


In [ ]:
# !pip install jsonlines

## 키값 추출

In [110]:
import json
import jsonlines

file_path = "../data/raw/eflaw.jsonl"  # 실제 데이터로 교체

def get_all_keys(d, parent=""):
    keys = []
    if isinstance(d, dict):
        for k, v in d.items():
            full_key = f"{parent}.{k}" if parent else k
            keys.append(full_key)
            keys.extend(get_all_keys(v, full_key))
    elif isinstance(d, list):
        for item in d:
            keys.extend(get_all_keys(item, parent))
    return keys

all_keys = set()
with jsonlines.open(file_path) as reader:
    for obj in reader:
        all_keys.update(get_all_keys(obj))

all_keys = sorted(all_keys)

with jsonlines.open("../data/raw/keys_output.jsonl", mode="w") as writer:
    for key in all_keys:
        writer.write({"key": key})

print(f"총 {len(all_keys)}개 키 추출 완료 → ../data/raw/keys_output.jsonl 저장됨")

총 110개 키 추출 완료 → ../data/raw/keys_output.jsonl 저장됨


# 2. 불필요 키 제거

In [20]:
import jsonlines

INPUT_PATH = "../data/raw/eflaw.jsonl"
OUTPUT_PATH = "../data/preprocessed/eflaw.jsonl"

# 최상위에서 제거할 키
DROP_TOP_KEYS = {
    "공동부령정보",
    "법령일련번호",
    "소관부처명",    # 기본정보.소관부처와 중복
    "소관부처코드",
    "자법타법여부",
    "현행연혁코드",
}

# 본문.법령 내부에서 제거할 키
DROP_BODY_KEYS = {
    "기본정보",    # 최상위 메타데이터와 중복
    "별표",        # 이미지/HWP 파일 참조
    "법령키",      # 내부 키
    "개정문",      # 개정 경위문
    "제개정이유",  # 제개정이유내용 포함
}

# 조문단위 내부에서 제거할 키
DROP_조문단위_KEYS = {
    "조문제개정유형",
    "조문이동이전",
    "조문이동이후",
    "조문변경여부",
}

# 부칙단위 내부에서 제거할 키
DROP_부칙단위_KEYS = {
    "부칙공포일자",
    "부칙공포번호",
}


def filter_조문단위(unit: dict) -> dict:
    return {k: v for k, v in unit.items() if k not in DROP_조문단위_KEYS}


def filter_부칙단위(unit: dict) -> dict:
    return {k: v for k, v in unit.items() if k not in DROP_부칙단위_KEYS}


def filter_record(record: dict) -> dict:
    filtered = {k: v for k, v in record.items() if k not in DROP_TOP_KEYS}

    try:
        법령 = filtered["본문"]["법령"]
        법령 = {k: v for k, v in 법령.items() if k not in DROP_BODY_KEYS}

        # 조문단위 필터링 (list / dict 모두 대응)
        if "조문" in 법령:
            조문단위 = 법령["조문"].get("조문단위")
            if isinstance(조문단위, list):
                법령["조문"]["조문단위"] = [filter_조문단위(u) for u in 조문단위]
            elif isinstance(조문단위, dict):
                법령["조문"]["조문단위"] = filter_조문단위(조문단위)

        # 부칙단위 필터링
        if "부칙" in 법령:
            부칙단위 = 법령["부칙"].get("부칙단위")
            if isinstance(부칙단위, list):
                법령["부칙"]["부칙단위"] = [filter_부칙단위(u) for u in 부칙단위]
            elif isinstance(부칙단위, dict):
                법령["부칙"]["부칙단위"] = filter_부칙단위(부칙단위)

        filtered["본문"]["법령"] = 법령
    except (KeyError, TypeError):
        pass

    return filtered


records = []
with jsonlines.open(INPUT_PATH) as reader:
    for obj in reader:
        records.append(filter_record(obj))

with jsonlines.open(OUTPUT_PATH, mode="w") as writer:
    writer.write_all(records)

print(f"{len(records)}건 처리 완료 → {OUTPUT_PATH}")

1707건 처리 완료 → ../data/preprocessed/eflaw.jsonl


### 데이터의 키값 jsonl 파일에 저장

In [92]:
file_path = "../data/preprocessed/eflaw.jsonl"  # 실제 데이터로 교체

all_keys = set()
with jsonlines.open(file_path) as reader:
    for obj in reader:
        all_keys.update(get_all_keys(obj))

all_keys = sorted(all_keys)

with jsonlines.open("../data/preprocessed/keys_output_preprocessed.jsonl", mode="w") as writer:
    for key in all_keys:
        writer.write({"key": key})

print(f"총 {len(all_keys)}개 키 추출 완료 → ../data/preprocessed/keys_output_preprocessed.jsonl 저장됨")

총 34개 키 추출 완료 → ../data/preprocessed/keys_output_preprocessed.jsonl 저장됨


# 3. 데이터 전처리

`data/preprocessed/eflaw.jsonl`을 pandas DataFrame으로 로드하여
컬럼별 목적을 분류하고, 불필요한 컬럼을 근거와 함께 제거한다.  
이후 중첩 구조(조문단위)를 행 단위로 펼쳐 임베딩에 적합한 평탄형 JSONL로 저장한다.


## 3-1. 데이터 로드 및 구조 파악

In [21]:
import re
import numpy as np
import pandas as pd

df = pd.read_json('../data/preprocessed/eflaw.jsonl', lines=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1707 entries, 0 to 1706
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   법령명한글   1707 non-null   str   
 1   법령구분명   1707 non-null   str   
 2   공포번호    1707 non-null   int64 
 3   제개정구분명  1707 non-null   str   
 4   id      1707 non-null   int64 
 5   법령ID    1707 non-null   int64 
 6   시행일자    1707 non-null   int64 
 7   공포일자    1707 non-null   int64 
 8   법령약칭명   1707 non-null   str   
 9   본문      1707 non-null   object
dtypes: int64(5), object(1), str(4)
memory usage: 133.5+ KB


In [94]:
# 최상위 컬럼 구조 확인
df.iloc[0]

법령명한글                                                건축물관리법
법령구분명                                                    법률
공포번호                                                  20549
제개정구분명                                                 타법개정
id                                                        1
법령ID                                                  13478
시행일자                                               20250604
공포일자                                               20241203
법령약칭명                                                      
본문        {'법령': {'부칙': {'부칙단위': [{'부칙키': '2019043016414...
Name: 0, dtype: object

In [95]:
# '본문' 컬럼 내부 구조 확인
df.iloc[0]['본문']['법령']['조문']['조문단위'][0]

{'조문번호': '1',
 '조문시행일자': '20250604',
 '조문키': '0001000',
 '조문내용': '                        제1장 총칙',
 '조문제목': '',
 '조문여부': '전문'}

## 3-2. 컬럼별 분류 및 처리 방침

**eflaw(현행법령) 항목별 분류**

1. `Metadata` — 검색 필터·출처 표시에 사용
   - `법령명한글` : 법령 공식 한글명. 임베딩 prefix로도 활용
   - `법령약칭명` : 법령 약칭. `법령명한글`이 없을 때 fallback으로 사용
   - `법령ID` : 법령 고유 식별자. chunk_id 생성에 사용
   - `법령구분명` : 법률 / 대통령령 / 부령 등 법령 종류. 검색 필터 활용
   - `시행일자` : 법령 시행 날짜. 버전 필터(최신 법령 우선)에 활용
   - `공포일자` : 법령 공포 날짜. 시행일자와 함께 법령 버전 추적에 사용
   - `제개정구분명` : 제정 / 일부개정 / 전부개정 구분. 개정 여부 판단에 활용

2. `Content` — 임베딩 텍스트 생성에 사용
   - `본문.법령.조문.조문단위` : 각 조문의 본문(조문내용, 항, 호, 목)
   - `본문.법령.부칙.부칙단위` : 부칙 내용

3. `제외` — 이미 제거됐거나 임베딩에 불필요한 항목
   - `법령상세링크` : 섹션 2에서 제거 완료. 링크는 검색 후 법령ID로 재구성 가능
   - `조문여부 == "전문"` : 장·절 구분자로 실제 조문 내용 없음 → 임베딩 불필요
   - `기본정보`, `별표`, `법령키`, `개정문`, `제개정이유` : 섹션 2에서 제거 완료
   - 개정이력 태그 `<개정 YYYY.MM.DD>` : 날짜 문자열이 임베딩 벡터 공간에서 노이즈 유발
   - 표 구분선(`─`, `│` 등, U+2500~U+257F) : 원본 HWP 표 구조 잔재. 의미 없음

In [22]:
# 항목별 분류에 따라 필요한 컬럼만 제거

df = df[['법령명한글', '법령약칭명', '법령ID', '법령구분명', '시행일자', '공포일자', '제개정구분명', '본문']]
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1707 entries, 0 to 1706
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   법령명한글   1707 non-null   str   
 1   법령약칭명   1707 non-null   str   
 2   법령ID    1707 non-null   int64 
 3   법령구분명   1707 non-null   str   
 4   시행일자    1707 non-null   int64 
 5   공포일자    1707 non-null   int64 
 6   제개정구분명  1707 non-null   str   
 7   본문      1707 non-null   object
dtypes: int64(3), object(1), str(4)
memory usage: 106.8+ KB


## 3-3. 조문 단위 평탄화 (explode)

eflaw 데이터는 1건 = 1개 법령이며, 각 법령 안에 수십~수백 개의 조문이 중첩되어 있다.  
RAG 검색 단위는 조문 1건이므로, `조문단위` 배열을 행(row) 단위로 펼친다.

**텍스트 추출 규칙**
- `조문내용` : 조문의 본문
- `항.항내용` : 항 단위 본문 (항이 있는 경우)
- `항.호.호내용` : 호 단위 본문 (호가 있는 경우)
- `항.호.목.목내용` : 목 단위 본문 (목이 있는 경우)
- 전문(장·절 구분자) : 조문 내용 없음 → 제외

조  :	Article
항  :	Paragraph
호	:	Subparagraph
목	:	Item

In [23]:
def _flatten_text(raw) -> str:
    """str | list 혼재 필드를 단일 문자열로 변환"""
    if not raw:
        return ''
    if isinstance(raw, str):
        return raw
    if isinstance(raw, list):
        return ' '.join(_flatten_text(x) for x in raw if x)
    return str(raw)


def _extract_text(조: dict) -> str:
    """조문 딕셔너리에서 전체 텍스트 추출 (조문내용 + 항·호·목)"""
    parts = []

    조문내용 = _flatten_text(조.get('조문내용', ''))
    if 조문내용:
        parts.append(조문내용)

    항_raw = 조.get('항', [])
    if isinstance(항_raw, dict):
        항_raw = [항_raw]
    for 항 in (항_raw or []):
        항내용 = _flatten_text(항.get('항내용', ''))
        if 항내용:
            parts.append(항내용)

        호_raw = 항.get('호', [])
        if isinstance(호_raw, dict):
            호_raw = [호_raw]
        for 호 in (호_raw or []):
            호내용 = _flatten_text(호.get('호내용', ''))
            if 호내용:
                parts.append(호내용)

            목_raw = 호.get('목', [])
            if isinstance(목_raw, dict):
                목_raw = [목_raw]
            for 목 in (목_raw or []):
                목내용 = _flatten_text(목.get('목내용', ''))
                if 목내용:
                    parts.append(목내용)

    return ' '.join(parts)


def _normalize(text: str) -> str:
    """
    텍스트 정규화
    - 표 구분선 제거 : HWP 표 잔재(─ │ 등)가 임베딩 노이즈 유발
    - 개정이력 태그 제거 : <개정 2022.2.3> 형식이 날짜 문자열 노이즈 유발
    - 연속 공백 정리
    """
    text = re.sub(r'[\u2500-\u257F]+', '', text)   # 표 구분선
    text = re.sub(r'<[^>]{1,150}>', '', text)        # 개정이력 태그
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def _format_article_no(조문번호, 조문가지번호) -> str:
    """조문번호 + 조문가지번호 → '제N조' 또는 '제N조의M'"""
    if not 조문번호:
        return ''
    try:
        base   = int(str(조문번호).strip())
        branch = int(str(조문가지번호).strip()) if 조문가지번호 else 0
        return f'제{base}조의{branch}' if branch else f'제{base}조'
    except ValueError:
        return str(조문번호)


records = []
for _, row in df.iterrows():
    법령명 = row['법령명한글'] or row.get('법령약칭명') or '알수없음'
    meta = {
        '법령명'     : 법령명,
        '법령ID'     : str(row['법령ID']),
        '법령구분명' : row['법령구분명'] or '',
        '시행일자'   : str(row['시행일자']) if pd.notna(row['시행일자']) else '',
        '공포일자'   : str(row['공포일자']) if pd.notna(row['공포일자']) else '',
        '제개정구분명': row['제개정구분명'] or '',
    }

    본문 = row['본문']
    법령 = 본문.get('법령', {}) if isinstance(본문, dict) else {}
    조문단위 = 법령.get('조문', {}).get('조문단위', [])
    if isinstance(조문단위, dict):
        조문단위 = [조문단위]

    for 조 in (조문단위 or []):
        if 조.get('조문여부') == '전문':   # 장·절 구분자 제외
            continue

        조문번호_표기 = _format_article_no(조.get('조문번호'), 조.get('조문가지번호'))
        조문제목      = _normalize(_flatten_text(조.get('조문제목', '')))
        raw_text      = _extract_text(조)
        text          = _normalize(raw_text)

        if not text:   # 텍스트 없는 빈 조문 제외
            continue

        records.append({
            **meta,
            '조문번호_표기' : 조문번호_표기,
            '조문제목'      : 조문제목,
            'text'          : f'[{법령명}] {조문번호_표기} {조문제목} {text}'.strip(),
        })

article_df = pd.DataFrame(records)
print(f'총 {len(article_df):,}개 조문')
article_df.info()

총 105,501개 조문
<class 'pandas.DataFrame'>
RangeIndex: 105501 entries, 0 to 105500
Data columns (total 9 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   법령명      105501 non-null  str  
 1   법령ID     105501 non-null  str  
 2   법령구분명    105501 non-null  str  
 3   시행일자     105501 non-null  str  
 4   공포일자     105501 non-null  str  
 5   제개정구분명   105501 non-null  str  
 6   조문번호_표기  105501 non-null  str  
 7   조문제목     105501 non-null  str  
 8   text     105501 non-null  str  
dtypes: str(9)
memory usage: 7.2 MB


## 3-4. 중복 확인 및 제거

동일 키워드로 수집된 법령이 중복 포함될 수 있으므로  
`법령ID` + `조문번호_표기` 조합으로 완전히 동일한 조문을 제거한다.  
`text` 기준으로도 중복 확인하여 개정 전·후 동일 내용 조문을 검토한다.

In [24]:
# 컬럼별 중복 수 확인
for col in article_df.columns:
    n = article_df[col].duplicated().sum()
    if n > 0:
        print(f'{col}: {n:,}개 중복')

법령명: 103,794개 중복
법령ID: 103,794개 중복
법령구분명: 105,470개 중복
시행일자: 105,108개 중복
공포일자: 105,085개 중복
제개정구분명: 105,497개 중복
조문번호_표기: 101,124개 중복
조문제목: 35,924개 중복


In [25]:
# text 완전 중복 확인 (동일 법령ID + 조문번호_표기 + 내용이 같은 경우)
dup_mask = article_df.duplicated(subset=['법령ID', '조문번호_표기', 'text'], keep=False)
print(f'완전 중복 행: {dup_mask.sum():,}개')
article_df[dup_mask].head(6)

완전 중복 행: 0개


,법령명,법령ID,법령구분명,시행일자,공포일자,제개정구분명,조문번호_표기,조문제목,text


In [100]:
# 완전 중복 제거 (법령ID + 조문번호_표기 + text 기준)
before = len(article_df)
article_df = article_df.drop_duplicates(subset=['법령ID', '조문번호_표기', 'text'], keep='first')
print(f'중복 제거: {before - len(article_df):,}개 제거 → {len(article_df):,}개 남음')

중복 제거: 0개 제거 → 105,501개 남음


## 3-5. 결과 저장 및 검증

In [27]:
OUTPUT_PATH = '../data/preprocessed/eflaw_flat.jsonl'

article_df.to_json(OUTPUT_PATH, orient='records', lines=True, force_ascii=False)
print(f'저장 완료: {len(article_df):,}개 조문 → {OUTPUT_PATH}')

저장 완료: 105,501개 조문 → ../data/preprocessed/eflaw_flat.jsonl


In [103]:
# 길이 분포 확인
lengths = article_df['text'].str.len()
print(f'text 길이 통계')
print(f'  최소: {lengths.min():,}자')
print(f'  최대: {lengths.max():,}자')
print(f'  평균: {lengths.mean():,.0f}자')
print(f'  중앙값: {lengths.median():,.0f}자')

# 주택/임대차 관련 법령
주택관련 = article_df[article_df['법령명'].str.contains('임대차|주택|전세', na=False)]
print(f'\n주택/임대차 관련 조문: {len(주택관련):,}개')
print(주택관련['법령명'].value_counts().head(10).to_string())

text 길이 통계
  최소: 16자
  최대: 22,750자
  평균: 441자
  중앙값: 297자

주택/임대차 관련 조문: 2,163개
법령명
주택법                125
공동주택관리법 시행령        124
주택법 시행령            124
공공주택 특별법           119
공동주택관리법            113
주택건설기준 등에 관한 규정     90
한국주택금융공사법           90
주택공급에 관한 규칙         89
민간임대주택에 관한 특별법      89
공공주택 특별법 시행령        87


In [104]:
# 샘플 확인
sample = article_df[article_df['법령명'].str.contains('주택임대차보호법', na=False)].head(3)
for _, row in sample.iterrows():
    print(f"[{row['법령명']}] {row['조문번호_표기']} {row['조문제목']}")
    print(f"  {row['text'][:200]}")
    print()

[주택임대차보호법] 제1조 목적
  [주택임대차보호법] 제1조 목적 제1조(목적) 이 법은 주거용 건물의 임대차(賃貸借)에 관하여 「민법」에 대한 특례를 규정함으로써 국민 주거생활의 안정을 보장함을 목적으로 한다.

[주택임대차보호법] 제2조 적용 범위
  [주택임대차보호법] 제2조 적용 범위 제2조(적용 범위) 이 법은 주거용 건물(이하 "주택"이라 한다)의 전부 또는 일부의 임대차에 관하여 적용한다. 그 임차주택(賃借住宅)의 일부가 주거 외의 목적으로 사용되는 경우에도 또한 같다.

[주택임대차보호법] 제3조 대항력 등
  [주택임대차보호법] 제3조 대항력 등 제3조(대항력 등) ① 임대차는 그 등기(登記)가 없는 경우에도 임차인(賃借人)이 주택의 인도(引渡)와 주민등록을 마친 때에는 그 다음 날부터 제삼자에 대하여 효력이 생긴다. 이 경우 전입신고를 한 때에 주민등록이 된 것으로 본다. ② 주택도시기금을 재원으로 하여 저소득층 무주택자에게 주거생활 안정을 목적으로 전세임대주



In [105]:
article_df.head(3)

,법령명,법령ID,법령구분명,시행일자,공포일자,제개정구분명,조문번호_표기,조문제목,text
0,건축물관리법,13478,법률,20250604,20241203,타법개정,제1조,목적,[건축물관리법] 제1조 목적 제1조(목적) 이 법은 건축물의 안전을 확보하고 편리ㆍ...
1,건축물관리법,13478,법률,20250604,20241203,타법개정,제2조,정의,[건축물관리법] 제2조 정의 제2조(정의) 이 법에서 사용하는 용어의 뜻은 다음과 ...
2,건축물관리법,13478,법률,20250604,20241203,타법개정,제3조,국가 및 지방자치단체의 책무,[건축물관리법] 제3조 국가 및 지방자치단체의 책무 제3조(국가 및 지방자치단체의 ...


# 4. 청킹 (Chunking)

`data/preprocessed/eflaw_flat.jsonl`의 `text` 필드를 청킹한다.

## 전략 비교

| 전략 | 방식 | chunk_size | overlap | 선택 근거 |
|------|------|-----------|---------|-----------|
| **A: RC-500** | RecursiveCharacterTextSplitter | 500자 | 50자 | 긴 조문(항·호·목 다수) 대응. |
| **B: Article** | 조문 1건 = 청크 1건 (분할 없음) | — | — | 조문이 법령 최소 의미 단위. 분할 시 문맥 손실 가능 |
| **C: RC-300** | RecursiveCharacterTextSplitter | 300자 | 30자 | 짧은 청크로 precision ↑ 여부 확인 (recall ↓ 트레이드오프) |

→ **4-6절에서 MRR@5 · Hit@3 기준으로 최종 전략 1개 선택**

---

### 공통 설정
- 메타데이터: 분할 후에도 부모 조문 메타 전체 상속
- `chunk_id`: `{법령ID}_{조문번호_표기}` (단일 청크) / `{법령ID}_{조문번호_표기}_{i}` (분할 시)
- separators: `'. '` → `', '` → `' '` → `''` 순
  (한국어 법령은 줄바꿈 정규화 완료 → 문장부호 기준 분할이 가장 자연스러움)

## 4-1. 설정 및 데이터 로드

In [28]:
# !pip install langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json
from pathlib import Path

FLAT_PATH   = Path('../data/preprocessed/eflaw_flat.jsonl')

CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50

flat_records = [json.loads(l) for l in FLAT_PATH.open(encoding='utf-8')]
print(f'로드: {len(flat_records):,}개 조문')

lengths = [len(r['text']) for r in flat_records]
print(f'text 길이  최소={min(lengths):,}  최대={max(lengths):,}  평균={sum(lengths)/len(lengths):,.0f}자')
print(f'{CHUNK_SIZE}자 초과 조문: {sum(1 for l in lengths if l > CHUNK_SIZE):,}개')

로드: 105,501개 조문
text 길이  최소=16  최대=22,750  평균=441자
500자 초과 조문: 29,803개


## 4-2. 전략 A: Article 단위 (분할 없음)

조문 1건을 청크 1건으로 그대로 사용한다.

**선택 근거**:
- 조문은 법령의 최소 의미 단위 (제N조 = 하나의 규정)
- 분할 시 '항'이 앞 조문내용과 분리되어 문맥 손실 발생 가능
- 단점: 항·호·목이 많은 조문은 500자 초과 → 임베딩 품질 저하 우려

In [29]:
# 전략 A: 분할 없이 조문 단위 그대로 청크화
# 단일 청크이므로 chunk_id에 인덱스(_i) 불필요
chunks_A = []
for record in flat_records:
    chunks_A.append({
        'chunk_id': f"{record['법령ID']}_{record['조문번호_표기']}",
        **{k: v for k, v in record.items() if k != 'text'},
        'text': record['text'],
    })

chunk_lengths_A = [len(c['text']) for c in chunks_A]
print(f'전략 A  청크 수: {len(chunks_A):,}개')
print(f'  최소: {min(chunk_lengths_A):,}자')
print(f'  최대: {max(chunk_lengths_A):,}자')
print(f'  평균: {sum(chunk_lengths_A)/len(chunk_lengths_A):,.0f}자')
print(f'  500자 초과: {sum(1 for l in chunk_lengths_A if l > 500):,}개  ← 임베딩 품질 저하 구간')

전략 A  청크 수: 105,501개
  최소: 16자
  최대: 22,750자
  평균: 441자
  500자 초과: 29,803개  ← 임베딩 품질 저하 구간


### 전략 A 저장 및 샘플 확인

In [31]:
from pathlib import Path

CHUNKS_A_PATH = Path('../data/preprocessed/eflaw_chunks.jsonl')

with CHUNKS_A_PATH.open('w', encoding='utf-8') as f:
    for chunk in chunks_A:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

print(f'저장 완료: {len(chunks_A):,}개 → {CHUNKS_A_PATH}')


저장 완료: 105,501개 → ../data/preprocessed/eflaw_chunks.jsonl


In [67]:
# 주택임대차보호법 조문만 필터링 후 인덱스로 원하는 샘플 선택
filtered_A = [c for c in chunks_A if '주택임대차보호법' in c['법령명']]
print(f'주택임대차보호법 청크 수: {len(filtered_A):,}개  (0 ~ {len(filtered_A)-1})')

idx = 76
print(f'\n[인덱스 {idx}]')
print(json.dumps(filtered_A[idx], ensure_ascii=False, indent=2))

주택임대차보호법 청크 수: 78개  (0 ~ 77)

[인덱스 76]
{
  "chunk_id": "4950_제34조",
  "법령명": "주택임대차보호법 시행령",
  "법령ID": "4950",
  "법령구분명": "대통령령",
  "시행일자": "20260102",
  "공포일자": "20251230",
  "제개정구분명": "타법개정",
  "조문번호_표기": "제34조",
  "조문제목": "조정서의 작성",
  "text": "[주택임대차보호법 시행령] 제34조 조정서의 작성 제34조(조정서의 작성) 법 제26조제4항에 따른 조정서에는 다음 각 호의 사항을 기재하고, 위원장 및 조정에 참여한 조정위원이 서명 또는 기명날인하여야 한다. 1. 사건번호 및 사건명 2. 당사자의 성명, 생년월일 및 주소(법인의 경우 명칭, 법인등록번호 및 본점의 소재지를 말한다) 3. 임차주택 소재지 4. 신청의 취지 및 이유 5. 조정내용(법 제26조제4항에 따라 강제집행을 승낙하는 취지의 합의를 포함한다) 6. 작성일"
}


## 4-3. 전략 B: RC-500 텍스트 분할


In [44]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['. ', ', ', ' ', ''],
)

chunks = []
for record in flat_records:
    sub_texts = splitter.split_text(record['text'])
    for i, text in enumerate(sub_texts):
        chunk_id = (
            f"{record['법령ID']}_{record['조문번호_표기']}_{i}"
            if len(sub_texts) > 1
            else f"{record['법령ID']}_{record['조문번호_표기']}"
        )
        chunks.append({
            'chunk_id': chunk_id,
            **{k: v for k, v in record.items() if k != 'text'},
            'text': text,
        })

print(f'원본 조문:  {len(flat_records):,}개')
print(f'분할 후:    {len(chunks):,}개 청크')
print(f'평균 분할:  {len(chunks)/len(flat_records):.2f}개/조문')

원본 조문:  105,501개
분할 후:    157,035개 청크
평균 분할:  1.49개/조문


## 4-4. 전략 B 결과 저장 및 검증

In [45]:
CHUNKS_PATH = Path('../data/preprocessed/eflaw_chunks_B.jsonl')

with CHUNKS_PATH.open('w', encoding='utf-8') as f:
    for chunk in chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

print(f'저장 완료: {len(chunks):,}개 청크 → {CHUNKS_PATH}')

# 길이 분포
chunk_lengths = [len(c['text']) for c in chunks]
print(f'\nchunk text 길이 통계')
print(f'  최소: {min(chunk_lengths):,}자')
print(f'  최대: {max(chunk_lengths):,}자')
print(f'  평균: {sum(chunk_lengths)/len(chunk_lengths):,.0f}자')
print(f'  {CHUNK_SIZE}자 초과: {sum(1 for l in chunk_lengths if l > CHUNK_SIZE):,}개')

저장 완료: 157,035개 청크 → ../data/preprocessed/eflaw_chunks_B.jsonl

chunk text 길이 통계
  최소: 4자
  최대: 500자
  평균: 301자
  500자 초과: 0개


In [59]:
# 샘플 청크 확인
# 주택임대차보호법 조문만 필터링 후 인덱스로 원하는 샘플 선택
filtered_B = [c for c in chunks if '주택임대차보호법' in c['법령명']]
print(f'주택임대차보호법 청크 수: {len(filtered_B):,}개  (0 ~ {len(filtered_B)-1})')

idx = 3  # ← 원하는 인덱스 번호로 변경
print(f'\n[인덱스 {idx}]')
print(json.dumps(filtered_B[idx], ensure_ascii=False, indent=2))

주택임대차보호법 청크 수: 100개  (0 ~ 99)

[인덱스 3]
{
  "chunk_id": "1248_제3조_1",
  "법령명": "주택임대차보호법",
  "법령ID": "1248",
  "법령구분명": "법률",
  "시행일자": "20260102",
  "공포일자": "20251001",
  "제개정구분명": "타법개정",
  "조문번호_표기": "제3조",
  "조문제목": "대항력 등",
  "text": ". 임대차가 끝나기 전에 그 직원이 변경된 경우에는 그 법인이 선정한 새로운 직원이 주택을 인도받고 주민등록을 마친 다음 날부터 제삼자에 대하여 효력이 생긴다. ④ 임차주택의 양수인(讓受人)(그 밖에 임대할 권리를 승계한 자를 포함한다)은 임대인(賃貸人)의 지위를 승계한 것으로 본다. ⑤ 이 법에 따라 임대차의 목적이 된 주택이 매매나 경매의 목적물이 된 경우에는 「민법」 제575조제1항ㆍ제3항 및 같은 법 제578조를 준용한다. ⑥ 제5항의 경우에는 동시이행의 항변권(抗辯權)에 관한 「민법」 제536조를 준용한다."
}


## 4-5. 전략 C: RC-300 텍스트 분할

chunk_size=300, chunk_overlap=30

**선택 근거**:
- 짧은 청크는 임베딩 벡터가 단일 의미에 집중 → precision(정밀도) ↑ 가능
- 단점: 너무 잘게 쪼개면 조문 내 맥락이 끊겨 recall(재현율) ↓
- B(500자)와 비교하여 실제 top-k 품질이 나은지 평가로 확인

In [47]:
# flat_records는 4-1절에서 이미 로드된 변수 재사용
CHUNK_SIZE_C    = 300
CHUNK_OVERLAP_C = 30

splitter_C = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE_C,
    chunk_overlap=CHUNK_OVERLAP_C,
    separators=['. ', ', ', ' ', ''],
)

chunks_C = []
for record in flat_records:
    sub_texts = splitter_C.split_text(record['text'])
    for i, text in enumerate(sub_texts):
        # 분할된 경우에만 _i 인덱스 부여
        chunk_id = (
            f"{record['법령ID']}_{record['조문번호_표기']}_{i}"
            if len(sub_texts) > 1
            else f"{record['법령ID']}_{record['조문번호_표기']}"
        )
        chunks_C.append({
            'chunk_id': chunk_id,
            **{k: v for k, v in record.items() if k != 'text'},
            'text': text,
        })

chunk_lengths_C = [len(c['text']) for c in chunks_C]
print(f'원본 조문:  {len(flat_records):,}개')
print(f'전략 C  청크 수: {len(chunks_C):,}개  (평균 {len(chunks_C)/len(flat_records):.2f}개/조문)')
print(f'  최소: {min(chunk_lengths_C):,}자')
print(f'  최대: {max(chunk_lengths_C):,}자')
print(f'  평균: {sum(chunk_lengths_C)/len(chunk_lengths_C):,.0f}자')
print(f'  {CHUNK_SIZE_C}자 초과: {sum(1 for l in chunk_lengths_C if l > CHUNK_SIZE_C):,}개')

원본 조문:  105,501개
전략 C  청크 수: 227,867개  (평균 2.16개/조문)
  최소: 3자
  최대: 300자
  평균: 207자
  300자 초과: 0개


### 전략 C 저장 및 샘플 확인

In [48]:
CHUNKS_C_PATH = Path('../data/preprocessed/eflaw_chunks_C.jsonl')

with CHUNKS_C_PATH.open('w', encoding='utf-8') as f:
    for chunk in chunks_C:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

print(f'저장 완료: {len(chunks_C):,}개 → {CHUNKS_C_PATH}')


저장 완료: 227,867개 → ../data/preprocessed/eflaw_chunks_C.jsonl


In [ ]:
# 주택임대차보호법 조문만 필터링 후 인덱스로 원하는 샘플 선택
filtered_C = [c for c in chunks_C if '주택임대차보호법' in c['법령명']]
print(f'주택임대차보호법 청크 수: {len(filtered_C):,}개  (0 ~ {len(filtered_C)-1})')

idx =   # ← 원하는 인덱스 번호로 변경
print(f'\n[인덱스 {idx}]')
print(json.dumps(filtered_C[idx], ensure_ascii=False, indent=2))

주택임대차보호법 청크 수: 140개  (0 ~ 139)

[인덱스 6]
{
  "chunk_id": "1248_제3조의2_1",
  "법령명": "주택임대차보호법",
  "법령ID": "1248",
  "법령구분명": "법률",
  "시행일자": "20260102",
  "공포일자": "20251001",
  "제개정구분명": "타법개정",
  "조문번호_표기": "제3조의2",
  "조문제목": "보증금의 회수",
  "text": ". ② 제3조제1항ㆍ제2항 또는 제3항의 대항요건(對抗要件)과 임대차계약증서(제3조제2항 및 제3항의 경우에는 법인과 임대인 사이의 임대차계약증서를 말한다)상의 확정일자(確定日字)를 갖춘 임차인은 「민사집행법」에 따른 경매 또는 「국세징수법」에 따른 공매(公賣)를 할 때에 임차주택(대지를 포함한다)의 환가대금(換價代金)에서 후순위권리자(後順位權利者)나 그 밖의 채권자보다 우선하여 보증금을 변제(辨濟)받을 권리가 있다. ③ 임차인은 임차주택을 양수인에게 인도하지 아니하면 제2항에 따른 보증금을 받을 수 없다"
}


- 청킹 총평 A 이외의 B와 C는 내용이 짤려있는 경우가 다수 보임

# 5. 유사도 검색 (OpenAI text-embedding-3-small)

`data/preprocessed/eflaw_chunks_A.jsonl` → OpenAI 임베딩 → Qdrant 저장 → 자연어 검색

In [1]:
# !pip install openai qdrant-client python-dotenv -q

import sys
import os
from pathlib import Path
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

from openai import OpenAI
from src.vectordb import QdrantStore


class OpenAIEmbedder:
    """Embedder 인터페이스를 유지하면서 OpenAI API로 임베딩."""

    def __init__(self, model: str = 'text-embedding-3-small'):
        self._client = OpenAI()
        self._model  = model
        self.vector_size = 1536  # text-embedding-3-small 차원

    def embed(self, texts: list[str]) -> list[list[float]]:
        MAX_CHARS = 6000  # ← 추가 (6000자 ≈ 4000~6000토큰, 안전 마진 확보)
        texts = [t[:MAX_CHARS] for t in texts]  # ← 추가
        response = self._client.embeddings.create(input=texts, model=self._model)
        return [item.embedding for item in response.data]

    def embed_question(self, text: str) -> list[float]:
        return self.embed([text])[0]


embedder = OpenAIEmbedder()
print(f'모델: {embedder._model}  차원: {embedder.vector_size}')

/Users/m/Desktop/Project3/SKN24_3rd_6team/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


모델: text-embedding-3-small  차원: 1536


## 5-1. 데이터 로드 및 저장

In [2]:
import json
import time

CHUNKS_PATH = Path('../data/preprocessed/eflaw_chunks_A.jsonl')
QDRANT_PATH = '../data/embedding'
COLLECTION  = 'eflaw_openai'
BATCH_SIZE  = 100  # OpenAI API 호출 단위

chunks = [json.loads(line) for line in CHUNKS_PATH.open(encoding='utf-8')]
print(f'로드: {len(chunks):,}개 청크')

store = QdrantStore(COLLECTION, embedder, QDRANT_PATH)

MAX_CHARS = 6000  # ← 10000에서 변경
texts = [c['text'][:MAX_CHARS] for c in chunks]
metadatas = [{k: v for k, v in c.items() if k != 'text'} for c in chunks]

# 배치 단위 저장 (OpenAI API 호출 횟수 분산)
for start in range(0, len(texts), BATCH_SIZE):
    store.add_docs(
        texts=texts[start:start+BATCH_SIZE],
        metadatas=metadatas[start:start+BATCH_SIZE],
    )
    print(f'  {min(start+BATCH_SIZE, len(texts)):,} / {len(texts):,}', end='\r')
    time.sleep(1) 

print(f'\n{len(texts):,}개 저장 완료')

로드: 105,501개 청크


/Users/m/Desktop/Project3/SKN24_3rd_6team/notebooks/../src/vectordb/store.py:74: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20100 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  self._client.upsert(collection_name=self._collection, points=points)


  105,501 / 105,501
105,501개 저장 완료


## 5-2. 유사도 검색

In [15]:
query = '계약 만료'

results = store.search(query, top_k=5)

for r in results:
    print(f"유사도: {r['score']:.4f}")
    print(f"법령명: {r.get('법령명', '없음')}  {r.get('조문번호_표기', '')}  {r.get('조문제목', '')}")
    print(f"내용: {r['text']}")
    print('---')

유사도: 0.4949
법령명: 상법  제84조  계약의 종료
내용: [상법] 제84조 계약의 종료 제84조(계약의 종료) 조합계약은 다음의 사유로 인하여 종료한다. 1. 영업의 폐지 또는 양도 2. 영업자의 사망 또는 성년후견개시 3. 영업자 또는 익명조합원의 파산
---
유사도: 0.4949
법령명: 상법  제84조  계약의 종료
내용: [상법] 제84조 계약의 종료 제84조(계약의 종료) 조합계약은 다음의 사유로 인하여 종료한다. 1. 영업의 폐지 또는 양도 2. 영업자의 사망 또는 성년후견개시 3. 영업자 또는 익명조합원의 파산
---
유사도: 0.4860
법령명: 화물자동차 운수사업법  제38조  책임보험계약등의 계약 종료일 통지 등
내용: [화물자동차 운수사업법] 제38조 책임보험계약등의 계약 종료일 통지 등 제38조(책임보험계약등의 계약 종료일 통지 등) ① 보험회사등은 자기와 책임보험계약등을 체결하고 있는 보험등 의무가입자에게 그 계약종료일 30일 전까지 그 계약이 끝난다는 사실을 알려야 한다. ② 보험회사등은 자기와 책임보험계약등을 체결한 보험등 의무가입자가 그 계약이 끝난 후 새로운 계약을 체결하지 아니하면 그 사실을 지체 없이 국토교통부장관에게 알려야 한다. ③ 제1항 및 제2항에 따른 통지의 방법ㆍ절차에 필요한 사항은 국토교통부령으로 정한다.
---
유사도: 0.4649
법령명: 민법  제535조  계약체결상의 과실
내용: [민법] 제535조 계약체결상의 과실 제535조(계약체결상의 과실) ①목적이 불능한 계약을 체결할 때에 그 불능을 알았거나 알 수 있었을 자는 상대방이 그 계약의 유효를 믿었음으로 인하여 받은 손해를 배상하여야 한다. 그러나 그 배상액은 계약이 유효함으로 인하여 생길 이익액을 넘지 못한다. ②전항의 규정은 상대방이 그 불능을 알았거나 알 수 있었을 경우에는 적용하지 아니한다.
---
유사도: 0.4639
법령명: 상법  제85조  계약종료의 효과
내용: [상법] 제85조 계약종료의 효과 제85조(계약종료의 효과) 조합계약

In [ ]:
from src.vectordb import Embedder, QdrantStore

embedder = Embedder()

# 2단계: QdrantStore 만들기 (저장소)
#   - "laws" 는 컬렉션(폴더) 이름입니다. 원하는 이름으로 바꿔주세요
store = QdrantStore("laws", embedder)

# 3단계: 텍스트 저장
store.add_docs(
    texts=[
        "임대인은 임차인에게 보증금을 반환해야 한다.",
        "임차인은 계약 만료 후 목적물을 반환해야 한다.",
    ],
    metadatas=[
        {"source": "주택임대차보호법", "조문": "제3조"},
        {"source": "주택임대차보호법", "조문": "제4조"},
    ],
)

# 4단계: 질문으로 검색
results = store.search("보증금을 돌려받으려면?", top_k=3)

# 5단계: 결과 출력
for r in results:
    print(r["score"], r["text"], r["조문"])